# Traveler Insights UPTC 2026 — Team BETO

**Integrantes:** Juan David Tejedor Medina y Miguel Gerardo Moreno  
**Roles rotativos:** datos, modelado, evaluación y documentación  
**Repositorio:** https://github.com/miguelgerardomoreno-commits/traveler-insights-uptc-2026-beto  
**Notebook del equipo en Kaggle:** https://www.kaggle.com/code/miguelmoreno17/team-beto  
**Perfil de Kaggle:** https://www.kaggle.com/miguelmoreno17

## Fase 0. Portada y resumen

**Resumen (máximo 200 palabras):** se completa al finalizar con el problema, modelo elegido, accuracy local, resultado del leaderboard y una limitación.

Este notebook implementa el ciclo CRISP-ML(Q), registra riesgos de calidad y genera el archivo de envío desde un modelo guardado.

## Fuentes, reglas y decisiones previas

- Tarea: clasificación de sentimiento de reseñas turísticas en negativo, neutral o positivo.
- Métrica oficial de Kaggle: **accuracy**.
- Métricas complementarias: precisión, recall, F1 por clase y F1 macro.
- Límite declarado: cinco envíos diarios y dos finales.
- Datos externos: permitidos si son públicos y gratuitos.
- Datos de competencia: uso académico y no comercial.
- Modelo principal exigido por la asignatura: Transformer preentrenado ajustado por el equipo.
- Baseline obligatorio: TF-IDF con regresión logística.

### Discrepancias documentadas

La descripción de Kaggle afirma que test.csv no tiene etiquetas, pero los archivos entregados contienen un test.csv etiquetado y un submission.csv sin etiquetas. Este notebook usa el contenido real:

- train.csv: desarrollo y selección de modelos.
- test.csv: evaluación local final, consultada una sola vez después de elegir el modelo.
- submission.csv: conjunto sin etiquetas para generar la entrega.

La página también mezcla nombres de clase con un ejemplo numérico. Se generan ambos formatos de envío; el archivo principal sigue el ejemplo numérico: negativo=0, neutral=1, positivo=2.

## Configuración reproducible

In [ ]:
# Si Kaggle no tiene alguna dependencia, descomentar:
# %pip install -q transformers sentencepiece captum

from pathlib import Path
from collections import Counter
import copy
import hashlib
import json
import os
import platform
import random
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
ID_COL = "ID"
TEXT_COL = "Comentario"
TARGET_COL = "Sentimiento"
RATING_COL = "Valoración_num"
LABEL2ID = {"negativo": 0, "neutral": 1, "positivo": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

In [ ]:
def localizar_datos():
    candidatos = [
        Path("/kaggle/input/traveler-insights-uptc-2026"),
        Path.cwd() / "Data",
        Path.cwd() / "data",
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidatos.extend(p.parent for p in kaggle_input.rglob("train.csv"))
    vistos = set()
    for carpeta in candidatos:
        carpeta = carpeta.resolve()
        if carpeta in vistos:
            continue
        vistos.add(carpeta)
        requeridos = [carpeta / n for n in ("train.csv", "test.csv", "submission.csv")]
        if all(p.exists() for p in requeridos):
            return carpeta
    encontrados = list(kaggle_input.rglob("*.csv")) if kaggle_input.exists() else []
    raise FileNotFoundError(
        "No se encontraron train.csv, test.csv y submission.csv en una misma carpeta. "
        f"CSV visibles: {encontrados}"
    )

DATA_DIR = localizar_datos()
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ARTIFACTS_DIR = WORK_DIR / "artifacts"
MODELS_DIR = WORK_DIR / "models"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_INPUT_PATH = DATA_DIR / "submission.csv"

env_info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seed": SEED,
    "data_dir": str(DATA_DIR),
}
print(json.dumps(env_info, indent=2, ensure_ascii=False))
(ARTIFACTS_DIR / "env_info.json").write_text(
    json.dumps(env_info, indent=2, ensure_ascii=False), encoding="utf-8"
)

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
submission_input = pd.read_csv(SUBMISSION_INPUT_PATH)

print("train:", train.shape)
print("test etiquetado:", test.shape)
print("submission sin etiqueta:", submission_input.shape)

required = {ID_COL, TEXT_COL, TARGET_COL}
for nombre, df in [("train", train), ("test", test), ("submission", submission_input)]:
    assert required.issubset(df.columns), f"Faltan columnas en {nombre}"
    assert df[ID_COL].is_unique, f"ID duplicado en {nombre}"
    assert df[TEXT_COL].notna().all(), f"Comentario vacío/NaN en {nombre}"

assert train[TARGET_COL].notna().all()
assert test[TARGET_COL].notna().all()
assert submission_input[TARGET_COL].isna().all()
assert set(train[TARGET_COL]) == set(LABEL2ID)
assert set(test[TARGET_COL]) == set(LABEL2ID)

display(train.head(3))

## Fase 1. Comprensión del negocio y de los datos

### Objetivo de negocio

Una entidad de turismo regional u operador turístico de Sucre usaría el modelo para priorizar la revisión de comentarios y detectar experiencias que requieren intervención. El modelo apoya la decisión de qué reseñas revisar primero; no toma decisiones automáticas sobre establecimientos.

### Objetivo de ML

Clasificación multiclase usando como entrada principal el texto Comentario y como salida Sentimiento.

### Criterios de éxito definidos antes del entrenamiento

- **Negocio:** identificar reseñas negativas con recall mayor que el baseline y entregar una lista auditable.
- **ML:** accuracy de validación del Transformer al menos 0,02 superior al baseline TF-IDF; reportar también F1 macro y recall negativo.
- **Económico:** latencia media menor de 100 ms por reseña en una T4, modelo menor de 1,5 GB y ejecución dentro de una sesión de Kaggle.

### Restricciones

GPU T4, tiempo de sesión limitado, máximo cinco envíos diarios, dos envíos finales y licencias compatibles con uso académico.

In [ ]:
def normalizar_para_auditoria(texto):
    return re.sub(r"\s+", " ", str(texto).strip().lower())

def resumen_calidad(df, nombre):
    return {
        "conjunto": nombre,
        "filas": len(df),
        "columnas": len(df.columns),
        "textos_vacios": int(df[TEXT_COL].fillna("").str.strip().eq("").sum()),
        "duplicados_texto": int(df[TEXT_COL].map(normalizar_para_auditoria).duplicated().sum()),
        "ids_duplicados": int(df[ID_COL].duplicated().sum()),
        "etiquetas_vacias": int(df[TARGET_COL].isna().sum()),
    }

display(pd.DataFrame([
    resumen_calidad(train, "train"),
    resumen_calidad(test, "test"),
    resumen_calidad(submission_input, "submission"),
]))

class_table = (
    train[TARGET_COL].value_counts()
    .rename("conteo").to_frame()
    .assign(proporcion=lambda x: x["conteo"] / len(train))
)
display(class_table)

majority_class = train[TARGET_COL].mode()[0]
majority_pred = np.repeat(majority_class, len(train))
print("Baseline mayoritario — accuracy:",
      round(accuracy_score(train[TARGET_COL], majority_pred), 4))
print("Baseline mayoritario — F1 macro:",
      round(f1_score(train[TARGET_COL], majority_pred, average="macro"), 4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
order = train[TARGET_COL].value_counts().index
sns.countplot(data=train, x=TARGET_COL, order=order, ax=axes[0])
axes[0].set_title("Figura 1. Distribución de clases")

train["n_palabras"] = train[TEXT_COL].str.split().str.len()
sns.histplot(train["n_palabras"], bins=40, ax=axes[1])
axes[1].set_title("Figura 2. Longitud en palabras")

sns.countplot(data=train, x="Sitio", hue=TARGET_COL, ax=axes[2])
axes[2].set_title("Figura 3. Clase por plataforma")
plt.tight_layout()
plt.show()

display(train.groupby(TARGET_COL)[TEXT_COL].agg(
    cantidad="size",
    caracteres_mediana=lambda s: s.str.len().median(),
    palabras_mediana=lambda s: s.str.split().str.len().median(),
))

In [ ]:
# Auditoría de duplicados y solapamiento
for df in (train, test, submission_input):
    df["_texto_norm"] = df[TEXT_COL].map(normalizar_para_auditoria)

sets = {
    "train": set(train["_texto_norm"]),
    "test": set(test["_texto_norm"]),
    "submission": set(submission_input["_texto_norm"]),
}
print("Solapamiento train-test:", len(sets["train"] & sets["test"]))
print("Solapamiento train-submission:", len(sets["train"] & sets["submission"]))
print("Solapamiento test-submission:", len(sets["test"] & sets["submission"]))

dup_labels = train.groupby("_texto_norm")[TARGET_COL].nunique()
print("Textos duplicados con etiquetas contradictorias:", int((dup_labels > 1).sum()))

# Auditoría simple de privacidad
patterns = {
    "correo": r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
    "telefono": r"(?<!\d)(?:\+?57\s*)?(?:\d[\s.-]*){7,10}(?!\d)",
    "url_en_comentario": r"https?://\S+|www\.\S+",
}
privacy_rows = []
for nombre, patron in patterns.items():
    privacy_rows.append({
        "patron": nombre,
        "train": int(train[TEXT_COL].str.contains(patron, regex=True, na=False).sum()),
        "test": int(test[TEXT_COL].str.contains(patron, regex=True, na=False).sum()),
        "submission": int(submission_input[TEXT_COL].str.contains(patron, regex=True, na=False).sum()),
    })
display(pd.DataFrame(privacy_rows))

In [ ]:
# Hallazgo crítico: asociación entre metadatos y etiqueta
rating_audit = (
    pd.concat([train.assign(split="train"), test.assign(split="test")])
    .groupby(["Sitio", RATING_COL])[TARGET_COL]
    .agg(conteo="size", etiquetas="nunique", valores=lambda s: ", ".join(sorted(s.unique())))
    .reset_index()
)
display(rating_audit)
assert rating_audit["etiquetas"].max() == 1

site_target = pd.crosstab(
    pd.concat([train, test])["Sitio"],
    pd.concat([train, test])[TARGET_COL],
    margins=True,
)
display(site_target)

print(
    "Conclusión QA: Sitio + Valoración_num determina la etiqueta en todos los "
    "registros etiquetados. Estas columnas se excluyen del modelo principal."
)

### Interpretación del EDA y riesgos

- La clase neutral domina el conjunto; accuracy debe acompañarse de F1 macro y recall negativo.
- Sitio y Valoración_num revelan la etiqueta de forma determinista. Usarlos produciría un atajo que no demuestra comprensión del texto.
- Existen comentarios exactos repetidos entre conjuntos. Se eliminan del entrenamiento los textos que aparecen en test o submission.
- Los duplicados internos se mantienen juntos mediante grupos de texto para impedir fuga entre train y validación.
- La variable Municipio presenta variantes por espacios; se normaliza solo para análisis por grupos.
- El texto conserva mayúsculas, tildes, puntuación y negaciones.

**QA de la fase:** riesgo → auditoría de distribución/duplicados/metadatos → tablas y aserciones anteriores.

## Fase 2. Ingeniería de datos

In [ ]:
def limpiar_texto(texto):
    texto = re.sub(r"https?://\S+|www\.\S+", " ", str(texto))
    return re.sub(r"\s+", " ", texto).strip()

for df in (train, test, submission_input):
    df["texto"] = df[TEXT_COL].map(limpiar_texto)
    df["texto_grupo"] = df["texto"].map(
        lambda x: hashlib.sha256(normalizar_para_auditoria(x).encode("utf-8")).hexdigest()
    )

# Protección adicional: no entrenar con textos que reaparecen en test o submission
protected_texts = set(test["_texto_norm"]) | set(submission_input["_texto_norm"])
train_dev = train.loc[~train["_texto_norm"].isin(protected_texts)].copy().reset_index(drop=True)
train_dev["label"] = train_dev[TARGET_COL].map(LABEL2ID).astype(int)
test["label"] = test[TARGET_COL].map(LABEL2ID).astype(int)

print("Train original:", len(train))
print("Train después de retirar solapamientos protegidos:", len(train_dev))

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_dev["fold"] = -1
for fold, (_, val_idx) in enumerate(splitter.split(
    train_dev,
    y=train_dev["label"],
    groups=train_dev["texto_grupo"],
)):
    train_dev.loc[val_idx, "fold"] = fold

FOLD_VAL = 0
tr_df = train_dev[train_dev["fold"] != FOLD_VAL].reset_index(drop=True)
va_df = train_dev[train_dev["fold"] == FOLD_VAL].reset_index(drop=True)

assert set(tr_df["texto_grupo"]).isdisjoint(set(va_df["texto_grupo"]))
assert set(tr_df[ID_COL]).isdisjoint(set(va_df[ID_COL]))
assert train_dev["label"].isin([0, 1, 2]).all()
assert (train_dev["texto"].str.len() > 0).all()
assert set(train_dev["_texto_norm"]).isdisjoint(set(test["_texto_norm"]))
assert set(train_dev["_texto_norm"]).isdisjoint(set(submission_input["_texto_norm"]))

print("Entrenamiento:", len(tr_df), "| Validación:", len(va_df))
display(pd.crosstab(train_dev["fold"], train_dev[TARGET_COL]))

### Selección del tokenizer y max_length

Se comparan dos checkpoints exigidos por la rúbrica:

| Checkpoint | Alcance | Licencia declarada | Justificación |
|---|---|---|---|
| dccuchile/bert-base-spanish-wwm-cased | Español, BERT-base | CC BY 4.0 como intención, con descargo sobre corpus | Especializado en español |
| FacebookAI/xlm-roberta-base | Multilingüe, 100 idiomas | MIT | Robustez ante variantes y contenido multilingüe |

Fuentes: tarjetas oficiales de los modelos en Hugging Face. Se utiliza max_length=192 porque cubre la gran mayoría de BETO con costo moderado; el porcentaje se recalcula para ambos tokenizers.

In [ ]:
CHECKPOINTS = {
    "BETO": "dccuchile/bert-base-spanish-wwm-cased",
    "XLM-R": "FacebookAI/xlm-roberta-base",
}
MAX_LENGTH = 192
TOKENIZERS = {}
token_rows = []

for alias, checkpoint in CHECKPOINTS.items():
    tok = AutoTokenizer.from_pretrained(checkpoint)
    TOKENIZERS[alias] = tok
    lengths = train_dev["texto"].map(
        lambda x: len(tok(x, add_special_tokens=True, truncation=False)["input_ids"])
    )
    for p in [50, 90, 95, 99]:
        token_rows.append({
            "modelo": alias,
            "estadistico": f"p{p}",
            "tokens": int(np.percentile(lengths, p)),
        })
    token_rows.append({
        "modelo": alias,
        "estadistico": f"% truncado a {MAX_LENGTH}",
        "tokens": round(100 * (lengths > MAX_LENGTH).mean(), 2),
    })

display(pd.DataFrame(token_rows))
assert 0 < MAX_LENGTH <= 512

In [ ]:
# Pesos de clase; la versión suavizada se ensaya como cambio controlado
classes = np.array([0, 1, 2])
weights_full = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=tr_df["label"].values,
)
weights_sqrt = np.sqrt(weights_full)

print("Pesos completos:", np.round(weights_full, 3))
print("Pesos raíz:", np.round(weights_sqrt, 3))

qa_phase2 = {
    "sin_ids_compartidos": set(tr_df[ID_COL]).isdisjoint(set(va_df[ID_COL])),
    "sin_textos_compartidos": set(tr_df["texto_grupo"]).isdisjoint(set(va_df["texto_grupo"])),
    "etiquetas_en_rango": train_dev["label"].isin([0, 1, 2]).all(),
    "sin_textos_vacios": (train_dev["texto"].str.len() > 0).all(),
    "semilla_fija": SEED,
}
display(pd.Series(qa_phase2, name="resultado_QA"))

## Fase 3. Ingeniería del modelo

### Baseline clásico

El baseline usa exclusivamente Comentario, el mismo split y la métrica oficial. No utiliza Sitio, Valoración ni otros metadatos.

In [ ]:
def metricas(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "precision_neg": p[0],
        "recall_neg": r[0],
        "f1_neg": f[0],
    }

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=40_000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        C=2.0,
        max_iter=2_000,
        class_weight="balanced",
        random_state=SEED,
    )),
])

start = time.perf_counter()
baseline.fit(tr_df["texto"], tr_df["label"])
baseline_time = time.perf_counter() - start
baseline_pred = baseline.predict(va_df["texto"])
baseline_metrics = metricas(va_df["label"], baseline_pred)

print(json.dumps(baseline_metrics, indent=2))
print("Tiempo de ajuste:", round(baseline_time, 2), "s")
print(classification_report(
    va_df["label"], baseline_pred,
    labels=[0, 1, 2],
    target_names=[ID2LABEL[i] for i in range(3)],
    zero_division=0,
))

In [ ]:
EXPERIMENT_LOG = ARTIFACTS_DIR / "experimentos.csv"
LOG_COLUMNS = [
    "run_id", "modelo", "cambio", "seed", "max_length", "learning_rate",
    "batch_size", "epocas_planeadas", "mejor_epoca", "pesos_clase",
    "accuracy", "f1_macro", "recall_neg", "tiempo_s", "dispositivo", "notas"
]

def registrar_experimento(row):
    if EXPERIMENT_LOG.exists():
        log = pd.read_csv(EXPERIMENT_LOG)
        log = log[log["run_id"] != row["run_id"]]
    else:
        log = pd.DataFrame(columns=LOG_COLUMNS)
    clean = {c: row.get(c, None) for c in LOG_COLUMNS}
    log = pd.concat([log, pd.DataFrame([clean])], ignore_index=True)
    log.to_csv(EXPERIMENT_LOG, index=False)
    return log

experiment_log = registrar_experimento({
    "run_id": "B00",
    "modelo": "TF-IDF + Regresión Logística",
    "cambio": "Baseline obligatorio",
    "seed": SEED,
    "accuracy": baseline_metrics["accuracy"],
    "f1_macro": baseline_metrics["f1_macro"],
    "recall_neg": baseline_metrics["recall_neg"],
    "tiempo_s": baseline_time,
    "dispositivo": "CPU",
    "notas": "Solo texto; class_weight=balanced",
})
display(experiment_log)

### Arquitectura y diseño experimental

Cada Transformer usa el encoder preentrenado completo y una cabeza lineal de clasificación de tres clases. No se congela el encoder: todos sus parámetros se ajustan. Se usa AdamW, scheduler lineal, warmup del 10 %, clipping de gradiente y early stopping.

Se registran cinco corridas contando el baseline. Entre corridas Transformer se modifica un factor principal:

1. BETO sin pesos.
2. XLM-R sin pesos: cambia el checkpoint.
3. BETO con pesos raíz: cambia la pérdida.
4. BETO con pesos raíz y learning rate mayor: cambia el learning rate.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length, with_labels=True):
        self.texts = frame["texto"].tolist()
        self.with_labels = with_labels
        self.labels = frame["label"].tolist() if with_labels else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in item.items()}
        if self.with_labels:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

@torch.no_grad()
def predecir_modelo(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        labels = batch.pop("labels", None)
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**inputs).logits.detach().cpu()
        all_logits.append(logits)
        if labels is not None:
            all_labels.append(labels)
    logits = torch.cat(all_logits).numpy()
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = logits.argmax(axis=1)
    labels = torch.cat(all_labels).numpy() if all_labels else None
    return labels, preds, probs

In [ ]:
def entrenar_experimento(cfg):
    set_seed(cfg["seed"])
    run_dir = MODELS_DIR / cfg["run_id"]
    run_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(cfg["checkpoint"])
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["checkpoint"],
        num_labels=3,
        label2id=LABEL2ID,
        id2label=ID2LABEL,
    ).to(DEVICE)

    train_ds = ReviewDataset(tr_df, tokenizer, cfg["max_length"])
    valid_ds = ReviewDataset(va_df, tokenizer, cfg["max_length"])
    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"], shuffle=True,
        num_workers=2, pin_memory=torch.cuda.is_available()
    )
    valid_loader = DataLoader(
        valid_ds, batch_size=cfg["eval_batch_size"], shuffle=False,
        num_workers=2, pin_memory=torch.cuda.is_available()
    )

    class_weights = None
    if cfg["weight_mode"] == "sqrt":
        class_weights = torch.tensor(weights_sqrt, dtype=torch.float32, device=DEVICE)
    elif cfg["weight_mode"] == "full":
        class_weights = torch.tensor(weights_full, dtype=torch.float32, device=DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"]
    )
    steps_per_epoch = int(np.ceil(len(train_loader) / cfg["grad_accum"]))
    total_steps = steps_per_epoch * cfg["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(cfg["warmup_ratio"] * total_steps),
        num_training_steps=total_steps,
    )

    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best_state, best_score, best_epoch = None, (-1, -1), 0
    patience_count = 0
    history = []
    start = time.perf_counter()

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_loss = 0.0

        for step, batch in enumerate(tqdm(train_loader, desc=f"{cfg['run_id']} e{epoch}")):
            labels = batch.pop("labels").to(DEVICE)
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(**inputs).logits
                loss = loss_fn(logits, labels) / cfg["grad_accum"]
            scaler.scale(loss).backward()
            train_loss += loss.item() * cfg["grad_accum"]

            if (step + 1) % cfg["grad_accum"] == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        y_val, p_val, prob_val = predecir_modelo(model, valid_loader)
        m = metricas(y_val, p_val)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss / len(train_loader),
            **m,
        })
        score = (m["accuracy"], m["f1_macro"])
        print(cfg["run_id"], history[-1])

        if score > best_score:
            best_score = score
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg["patience"]:
                break

    elapsed = time.perf_counter() - start
    model.load_state_dict(best_state)
    model.save_pretrained(run_dir)
    tokenizer.save_pretrained(run_dir)
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(run_dir / "history.csv", index=False)

    y_val, p_val, prob_val = predecir_modelo(model, valid_loader)
    best_metrics = metricas(y_val, p_val)
    result = {
        "config": cfg,
        "metrics": best_metrics,
        "history": hist_df,
        "best_epoch": best_epoch,
        "duration": elapsed,
        "val_pred": p_val,
        "val_prob": prob_val,
        "model_dir": str(run_dir),
    }

    del model, train_loader, valid_loader, train_ds, valid_ds, best_state
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

In [ ]:
TRANSFORMER_CONFIGS = [
    {
        "run_id": "T01", "alias": "BETO", "checkpoint": CHECKPOINTS["BETO"],
        "change": "Primer Transformer; sin pesos de clase",
        "seed": SEED, "max_length": MAX_LENGTH, "learning_rate": 2e-5,
        "weight_decay": 0.01, "warmup_ratio": 0.10,
        "batch_size": 8, "eval_batch_size": 16, "grad_accum": 2,
        "epochs": 4, "patience": 1, "weight_mode": "none",
    },
    {
        "run_id": "T02", "alias": "XLM-R", "checkpoint": CHECKPOINTS["XLM-R"],
        "change": "Cambia solo el checkpoint respecto a T01",
        "seed": SEED, "max_length": MAX_LENGTH, "learning_rate": 2e-5,
        "weight_decay": 0.01, "warmup_ratio": 0.10,
        "batch_size": 4, "eval_batch_size": 8, "grad_accum": 4,
        "epochs": 4, "patience": 1, "weight_mode": "none",
    },
    {
        "run_id": "T03", "alias": "BETO", "checkpoint": CHECKPOINTS["BETO"],
        "change": "Añade pesos de clase suavizados respecto a T01",
        "seed": SEED, "max_length": MAX_LENGTH, "learning_rate": 2e-5,
        "weight_decay": 0.01, "warmup_ratio": 0.10,
        "batch_size": 8, "eval_batch_size": 16, "grad_accum": 2,
        "epochs": 4, "patience": 1, "weight_mode": "sqrt",
    },
    {
        "run_id": "T04", "alias": "BETO", "checkpoint": CHECKPOINTS["BETO"],
        "change": "Learning rate 3e-5 respecto a T03",
        "seed": SEED, "max_length": MAX_LENGTH, "learning_rate": 3e-5,
        "weight_decay": 0.01, "warmup_ratio": 0.10,
        "batch_size": 8, "eval_batch_size": 16, "grad_accum": 2,
        "epochs": 4, "patience": 1, "weight_mode": "sqrt",
    },
]

RUN_RESULTS = {}
for cfg in TRANSFORMER_CONFIGS:
    result = entrenar_experimento(cfg)
    RUN_RESULTS[cfg["run_id"]] = result
    m = result["metrics"]
    experiment_log = registrar_experimento({
        "run_id": cfg["run_id"],
        "modelo": cfg["checkpoint"],
        "cambio": cfg["change"],
        "seed": cfg["seed"],
        "max_length": cfg["max_length"],
        "learning_rate": cfg["learning_rate"],
        "batch_size": cfg["batch_size"],
        "epocas_planeadas": cfg["epochs"],
        "mejor_epoca": result["best_epoch"],
        "pesos_clase": cfg["weight_mode"],
        "accuracy": m["accuracy"],
        "f1_macro": m["f1_macro"],
        "recall_neg": m["recall_neg"],
        "tiempo_s": result["duration"],
        "dispositivo": str(DEVICE),
        "notas": "Encoder completo ajustado",
    })

display(experiment_log.sort_values(["accuracy", "f1_macro"], ascending=False))

In [ ]:
# Curvas y selección por métrica oficial; F1 macro desempata
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for run_id, result in RUN_RESULTS.items():
    h = result["history"]
    axes[0].plot(h["epoch"], h["train_loss"], marker="o", label=run_id)
    axes[1].plot(h["epoch"], h["accuracy"], marker="o", label=run_id)
axes[0].set_title("Figura 4. Pérdida de entrenamiento")
axes[1].set_title("Figura 5. Accuracy de validación")
for ax in axes:
    ax.set_xlabel("Época")
    ax.legend()
plt.tight_layout()
plt.show()

BEST_RUN_ID = max(
    RUN_RESULTS,
    key=lambda rid: (
        RUN_RESULTS[rid]["metrics"]["accuracy"],
        RUN_RESULTS[rid]["metrics"]["f1_macro"],
    ),
)
BEST_RESULT = RUN_RESULTS[BEST_RUN_ID]
BEST_CONFIG = BEST_RESULT["config"]
print("Mejor corrida:", BEST_RUN_ID)
print(json.dumps(BEST_RESULT["metrics"], indent=2))
print("Mejor época:", BEST_RESULT["best_epoch"])
print("¿Supera baseline + 0.02?:",
      BEST_RESULT["metrics"]["accuracy"] >= baseline_metrics["accuracy"] + 0.02)

### QA de la fase 3

- Todos los modelos usan el mismo split y métricas.
- Cada corrida tiene un ID y un cambio principal.
- Las semillas, versiones, hardware, duración e hiperparámetros quedan registrados.
- El modelo se selecciona con validación local, no con el leaderboard.
- El test etiquetado continúa reservado hasta después de la selección.
- El riesgo de sobreajuste se controla con early stopping y curvas.

## Fase 4. Evaluación del modelo

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(
    BEST_RESULT["model_dir"]
).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(BEST_RESULT["model_dir"])

valid_loader = DataLoader(
    ReviewDataset(va_df, best_tokenizer, BEST_CONFIG["max_length"]),
    batch_size=BEST_CONFIG["eval_batch_size"],
    shuffle=False,
)
y_val, pred_val, prob_val = predecir_modelo(best_model, valid_loader)

print(classification_report(
    y_val, pred_val,
    labels=[0, 1, 2],
    target_names=[ID2LABEL[i] for i in range(3)],
    zero_division=0,
))

cm = confusion_matrix(y_val, pred_val, labels=[0, 1, 2])
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=[ID2LABEL[i] for i in range(3)],
    yticklabels=[ID2LABEL[i] for i in range(3)],
)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title(f"Figura 6. Matriz de confusión — {BEST_RUN_ID}")
plt.show()

In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, confidence=0.95, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 3:
            continue
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    alpha = 1 - confidence
    return np.quantile(scores, [alpha / 2, 1 - alpha / 2])

acc_ci = bootstrap_ci(y_val, pred_val, accuracy_score)
f1_ci = bootstrap_ci(
    y_val, pred_val,
    lambda a, b: f1_score(a, b, average="macro")
)
print("IC 95% accuracy:", np.round(acc_ci, 4))
print("IC 95% F1 macro:", np.round(f1_ci, 4))

In [ ]:
# Análisis manual requerido de al menos 20 errores
error_df = va_df[[ID_COL, TEXT_COL, TARGET_COL, "Sitio", "Municipio"]].copy()
error_df["prediccion"] = [ID2LABEL[int(x)] for x in pred_val]
error_df["confianza"] = prob_val.max(axis=1)
error_df = error_df[error_df[TARGET_COL] != error_df["prediccion"]].copy()
error_df = error_df.sort_values("confianza", ascending=False)
error_df["categoria_manual"] = ""
error_df["hipotesis_causa"] = ""
error_df["accion_mitigacion"] = ""

errors_20 = error_df.head(20)
ERROR_PATH = ARTIFACTS_DIR / "analisis_20_errores.csv"
errors_20.to_csv(ERROR_PATH, index=False)

display(errors_20)
print("Completar manualmente las tres columnas finales antes de entregar.")
print("Categorías sugeridas: negación, ironía, reseña mixta, regionalismo, "
      "ortografía, texto insuficiente o posible ruido de etiqueta.")
print("Archivo:", ERROR_PATH)

In [ ]:
# Evaluación final única sobre test.csv, después de seleccionar el modelo
test_loader = DataLoader(
    ReviewDataset(test.assign(texto=test["texto"]), best_tokenizer, BEST_CONFIG["max_length"]),
    batch_size=BEST_CONFIG["eval_batch_size"],
    shuffle=False,
)
y_test, pred_test, prob_test = predecir_modelo(best_model, test_loader)
test_metrics = metricas(y_test, pred_test)
print("Métricas finales en test reservado:")
print(json.dumps(test_metrics, indent=2))

print(classification_report(
    y_test, pred_test,
    labels=[0, 1, 2],
    target_names=[ID2LABEL[i] for i in range(3)],
    zero_division=0,
))

In [ ]:
# Robustez: ruido ortográfico determinista y textos largos
def agregar_ruido(texto, seed=SEED, rate=0.08):
    rng = random.Random(seed + len(texto))
    words = str(texto).split()
    out = []
    for word in words:
        if len(word) > 4 and rng.random() < rate:
            i = rng.randint(1, len(word) - 2)
            chars = list(word)
            chars[i], chars[i + 1] = chars[i + 1], chars[i]
            word = "".join(chars)
        out.append(word)
    return " ".join(out)

robust_df = va_df.copy()
robust_df["texto"] = robust_df["texto"].map(agregar_ruido)
robust_loader = DataLoader(
    ReviewDataset(robust_df, best_tokenizer, BEST_CONFIG["max_length"]),
    batch_size=BEST_CONFIG["eval_batch_size"],
    shuffle=False,
)
_, pred_noisy, _ = predecir_modelo(best_model, robust_loader)
noisy_metrics = metricas(va_df["label"], pred_noisy)

long_mask = va_df["texto"].str.split().str.len() >= va_df["texto"].str.split().str.len().quantile(.90)
long_metrics = metricas(
    va_df.loc[long_mask, "label"],
    pred_val[long_mask.to_numpy()],
) if long_mask.sum() else {}

robustness = pd.DataFrame([
    {"escenario": "validación original", **metricas(y_val, pred_val)},
    {"escenario": "ruido ortográfico", **noisy_metrics},
    {"escenario": "10% textos más largos", **long_metrics},
])
display(robustness)

In [ ]:
# Desempeño responsable por grupos con tamaño suficiente
group_rows = []
eval_groups = va_df.copy()
eval_groups["pred"] = pred_val
eval_groups["Municipio_norm"] = eval_groups["Municipio"].astype(str).str.strip()

for field in ["Sitio", "Municipio_norm"]:
    for value, group in eval_groups.groupby(field):
        if len(group) < 8:
            continue
        m = metricas(group["label"], group["pred"])
        group_rows.append({"dimension": field, "grupo": value, "n": len(group), **m})

group_metrics = pd.DataFrame(group_rows)
display(group_metrics.sort_values(["dimension", "accuracy"]))

print("Interpretación responsable: diferencias pequeñas con n reducido no prueban sesgo. "
      "Se reportan como señales para monitoreo, no como conclusiones causales.")

In [ ]:
# Explicación por oclusión de palabras para un ejemplo
@torch.no_grad()
def explicar_por_occlusion(texto, model, tokenizer, max_length, top_k=12):
    model.eval()
    words = texto.split()
    if not words:
        return pd.DataFrame()
    def score(texts):
        enc = tokenizer(
            texts, padding=True, truncation=True, max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)
        return torch.softmax(model(**enc).logits, dim=1).cpu().numpy()
    base = score([texto])[0]
    cls = int(base.argmax())
    mask = tokenizer.mask_token or tokenizer.unk_token
    variants = [" ".join(words[:i] + [mask] + words[i+1:]) for i in range(len(words))]
    altered = score(variants)[:, cls]
    return (
        pd.DataFrame({"palabra": words, "atribucion_occlusion": base[cls] - altered})
        .sort_values("atribucion_occlusion", ascending=False)
        .head(top_k)
    )

example_text = va_df.iloc[0]["texto"]
print("Texto:", example_text)
display(explicar_por_occlusion(
    example_text, best_model, best_tokenizer, BEST_CONFIG["max_length"]
))
print("Advertencia: la oclusión muestra sensibilidad local del modelo; no es una explicación causal.")

### Decisión frente a los criterios de éxito

La siguiente celda determina automáticamente el cumplimiento ML. La decisión de negocio debe considerar especialmente recall negativo, incertidumbre, robustez y errores manuales. Un buen leaderboard no compensa una decisión insegura sobre la clase minoritaria.

In [ ]:
decision = {
    "accuracy_transformer": BEST_RESULT["metrics"]["accuracy"],
    "accuracy_baseline": baseline_metrics["accuracy"],
    "margen_observado": BEST_RESULT["metrics"]["accuracy"] - baseline_metrics["accuracy"],
    "cumple_margen_0_02": (
        BEST_RESULT["metrics"]["accuracy"] >= baseline_metrics["accuracy"] + 0.02
    ),
    "recall_neg_transformer": BEST_RESULT["metrics"]["recall_neg"],
    "recall_neg_baseline": baseline_metrics["recall_neg"],
    "mejora_recall_neg": (
        BEST_RESULT["metrics"]["recall_neg"] > baseline_metrics["recall_neg"]
    ),
}
display(pd.Series(decision))

## Fase 5. Despliegue adaptado

Se vuelve a entrenar la configuración elegida usando todo train_dev, sin usar las etiquetas de test. El número de épocas se fija en la mejor época observada. Después se guarda el modelo y se generan los archivos de envío.

In [ ]:
# Liberar el modelo de evaluación antes del ajuste final para evitar falta de memoria.
for nombre in ["best_model", "valid_loader", "test_loader", "robust_loader"]:
    if nombre in globals():
        del globals()[nombre]
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
def entrenar_modelo_final(cfg, epochs):
    set_seed(cfg["seed"])
    tokenizer = AutoTokenizer.from_pretrained(cfg["checkpoint"])
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["checkpoint"], num_labels=3,
        label2id=LABEL2ID, id2label=ID2LABEL
    ).to(DEVICE)

    ds = ReviewDataset(train_dev, tokenizer, cfg["max_length"])
    loader = DataLoader(
        ds, batch_size=cfg["batch_size"], shuffle=True,
        num_workers=2, pin_memory=torch.cuda.is_available()
    )

    cw = None
    if cfg["weight_mode"] == "sqrt":
        all_w = compute_class_weight("balanced", classes=classes, y=train_dev["label"])
        cw = torch.tensor(np.sqrt(all_w), dtype=torch.float32, device=DEVICE)
    elif cfg["weight_mode"] == "full":
        all_w = compute_class_weight("balanced", classes=classes, y=train_dev["label"])
        cw = torch.tensor(all_w, dtype=torch.float32, device=DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"]
    )
    steps_per_epoch = int(np.ceil(len(loader) / cfg["grad_accum"]))
    total_steps = steps_per_epoch * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(cfg["warmup_ratio"] * total_steps), total_steps
    )
    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tqdm(loader, desc=f"final e{epoch+1}")):
            labels = batch.pop("labels").to(DEVICE)
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = loss_fn(model(**inputs).logits, labels) / cfg["grad_accum"]
            scaler.scale(loss).backward()
            if (step + 1) % cfg["grad_accum"] == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

    final_dir = MODELS_DIR / "final_model"
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    return model, tokenizer, final_dir

final_model, final_tokenizer, FINAL_MODEL_DIR = entrenar_modelo_final(
    BEST_CONFIG, BEST_RESULT["best_epoch"]
)
print("Modelo final guardado en:", FINAL_MODEL_DIR)

In [ ]:
# Inferencia reproducible y medición de costo
submission_frame = submission_input.copy()
submission_frame["texto"] = submission_frame[TEXT_COL].map(limpiar_texto)
submission_ds = ReviewDataset(
    submission_frame.assign(label=0),
    final_tokenizer,
    BEST_CONFIG["max_length"],
    with_labels=False,
)
submission_loader = DataLoader(
    submission_ds,
    batch_size=BEST_CONFIG["eval_batch_size"],
    shuffle=False,
)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
_, submission_pred, submission_prob = predecir_modelo(final_model, submission_loader)
elapsed = time.perf_counter() - start

latency_ms = 1000 * elapsed / len(submission_frame)
model_size_mb = sum(
    p.stat().st_size for p in FINAL_MODEL_DIR.rglob("*") if p.is_file()
) / (1024 ** 2)
peak_vram_mb = (
    torch.cuda.max_memory_allocated() / (1024 ** 2)
    if torch.cuda.is_available() else np.nan
)

cost_report = {
    "ejemplos": len(submission_frame),
    "tiempo_total_s": elapsed,
    "latencia_media_ms": latency_ms,
    "tamano_modelo_mb": model_size_mb,
    "pico_vram_mb": peak_vram_mb,
}
print(json.dumps(cost_report, indent=2))

In [ ]:
# Formato principal según el ejemplo numérico del Overview
submission_numeric = pd.DataFrame({
    ID_COL: submission_frame[ID_COL],
    TARGET_COL: submission_pred.astype(int),
})
NUMERIC_PATH = WORK_DIR / "submission_kaggle.csv"
submission_numeric.to_csv(NUMERIC_PATH, index=False)

# Alternativa por la contradicción textual de la página
submission_labels = pd.DataFrame({
    ID_COL: submission_frame[ID_COL],
    TARGET_COL: [ID2LABEL[int(x)] for x in submission_pred],
})
LABEL_PATH = WORK_DIR / "submission_etiquetas.csv"
submission_labels.to_csv(LABEL_PATH, index=False)

assert list(submission_numeric.columns) == [ID_COL, TARGET_COL]
assert len(submission_numeric) == len(submission_frame)
assert submission_numeric[ID_COL].is_unique
assert submission_numeric[TARGET_COL].isin([0, 1, 2]).all()

print("Archivo principal:", NUMERIC_PATH)
print("Archivo alternativo:", LABEL_PATH)
display(submission_numeric.head())
display(submission_numeric[TARGET_COL].value_counts().sort_index())

In [ ]:
def clasificar_resena(texto):
    texto = limpiar_texto(texto)
    if not texto:
        raise ValueError("La reseña no puede estar vacía.")
    enc = final_tokenizer(
        texto, max_length=BEST_CONFIG["max_length"],
        truncation=True, return_tensors="pt"
    ).to(DEVICE)
    final_model.eval()
    with torch.no_grad():
        prob = torch.softmax(final_model(**enc).logits, dim=1)[0].cpu().numpy()
    pred = int(prob.argmax())
    return {
        "sentimiento": ID2LABEL[pred],
        "confianza": float(prob[pred]),
        "probabilidades": {ID2LABEL[i]: float(prob[i]) for i in range(3)},
    }

display(clasificar_resena(
    "El lugar es hermoso, pero la atención fue lenta y poco amable."
))

## Fase 6. Monitoreo y mantenimiento

Se monitorean distribución de predicciones, confianza, longitud, vocabulario/UNK, plataforma y municipio. La prueba de ruido ortográfico anterior constituye una simulación controlada de drift.

In [ ]:
monitoring_reference = {
    "class_distribution_train": train_dev[TARGET_COL].value_counts(normalize=True).to_dict(),
    "class_distribution_submission_pred": pd.Series(
        [ID2LABEL[int(x)] for x in submission_pred]
    ).value_counts(normalize=True).to_dict(),
    "mean_confidence_submission": float(submission_prob.max(axis=1).mean()),
    "median_words_train": float(train_dev["texto"].str.split().str.len().median()),
    "median_words_submission": float(submission_frame["texto"].str.split().str.len().median()),
    "accuracy_drop_typo_noise": float(
        BEST_RESULT["metrics"]["accuracy"] - noisy_metrics["accuracy"]
    ),
}
display(pd.Series(monitoring_reference))

RETRAIN_CRITERIA = {
    "caida_accuracy": 0.05,
    "cambio_distribucion_clase": 0.15,
    "caida_confianza_media": 0.10,
    "responsable": "Team BETO",
}
print(json.dumps(RETRAIN_CRITERIA, indent=2, ensure_ascii=False))

## Bitácora de IA generativa y contribuciones

La IA se utilizó como tutor, apoyo de diagnóstico y asistencia de programación. Todo código debe ser comprendido y validado por los integrantes antes de la defensa.

In [ ]:
AI_LOG_PATH = ARTIFACTS_DIR / "bitacora_ia.csv"
ai_log = pd.DataFrame([
    {
        "herramienta": "ChatGPT/Codex",
        "proposito": "Analizar rúbrica y reglas",
        "prompt_resumido": "Revisar criterios CRISP-ML(Q), Kaggle y riesgos",
        "aceptado": "Estructura por fases y checklist",
        "corregido": "Métrica oficial y roles reales de archivos",
        "validacion": "Contraste con Overview, Rules, rúbrica y CSV",
    },
    {
        "herramienta": "ChatGPT/Codex",
        "proposito": "Diagnosticar errores de Kaggle",
        "prompt_resumido": "Revisar fallos de Hugging Face y rutas",
        "aceptado": "Activar Internet y localizar datos automáticamente",
        "corregido": "Ruta rígida /kaggle/input/competitions",
        "validacion": "Ejecución limpia desde kernel reiniciado",
    },
    {
        "herramienta": "ChatGPT/Codex",
        "proposito": "Asistencia de programación",
        "prompt_resumido": "Construir pipeline reproducible según rúbrica",
        "aceptado": "Baseline, experimentos, QA, evaluación y despliegue",
        "corregido": "Revisión manual del equipo antes de entregar",
        "validacion": "Aserciones, métricas, logs y revisión del repositorio",
    },
])
ai_log.to_csv(AI_LOG_PATH, index=False)
display(ai_log)

### Contribución individual

| Integrante | Datos | Modelado | Evaluación | Documentación | Evidencia |
|---|---|---|---|---|---|
| Juan David Tejedor Medina | PENDIENTE | PENDIENTE | PENDIENTE | PENDIENTE | Commits/bitácora |
| Miguel Gerardo Moreno | PENDIENTE | PENDIENTE | PENDIENTE | PENDIENTE | Commits/bitácora |

## Cierre

### Conclusiones

Completar después de ejecutar todas las celdas: resultado frente al baseline, recall negativo, robustez y decisión de cumplimiento.

### Limitaciones

- Etiqueta fuertemente asociada con plataforma y valoración global.
- Clase negativa extremadamente escasa.
- Posibles discrepancias entre sentimiento textual y etiqueta.
- Pocos datos para estimar incertidumbre por subgrupo.
- La generalización fuera de los destinos y plataformas observados es incierta.

### Trabajo futuro

Reetiquetado manual de una muestra, ampliación de destinos, evaluación temporal, calibración de confianza y compresión del modelo.

## Referencias

- Studer, S. et al. (2021). Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology.
- Cañete, J. et al. (2020). Spanish Pre-Trained BERT Model and Evaluation Data.
- Conneau, A. et al. (2020). Unsupervised Cross-lingual Representation Learning at Scale.
- Devlin, J. et al. (2019). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.
- Vaswani, A. et al. (2017). Attention Is All You Need.
- Kaggle. Traveler Insights UPTC 2026: Overview, Data and Rules.
- Dataset of Tourist Profiles in Sucre. Zenodo: https://doi.org/10.5281/zenodo.13240643
- BETO model card: https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased
- XLM-R model card: https://huggingface.co/FacebookAI/xlm-roberta-base

## Lista final de entrega

- [ ] Ejecutar Restart & Run All sin errores.
- [ ] Confirmar que experimentos.csv contiene B00 y T01–T04.
- [ ] Completar manualmente analisis_20_errores.csv.
- [ ] Completar resumen de máximo 200 palabras.
- [ ] Completar contribuciones y roles rotativos.
- [ ] Registrar score público de Kaggle.
- [ ] Confirmar si Kaggle acepta submission_kaggle.csv numérico.
- [ ] Guardar una versión del notebook con salidas.
- [ ] Publicar notebook o compartirlo con el docente.
- [ ] Preparar PDF CRISP-ML(Q) de 10–15 páginas.
- [ ] Preparar defensa oral de 20 minutos.

In [ ]:
# Verificación automática final antes de guardar la versión
required_artifacts = [
    EXPERIMENT_LOG,
    ERROR_PATH,
    AI_LOG_PATH,
    NUMERIC_PATH,
    LABEL_PATH,
    FINAL_MODEL_DIR,
]
for path in required_artifacts:
    assert Path(path).exists(), f"Falta artefacto: {path}"

log_check = pd.read_csv(EXPERIMENT_LOG)
assert log_check["run_id"].nunique() >= 5, "Faltan cinco corridas trazables"
assert len(errors_20) >= min(20, len(error_df)), "Falta tabla de errores"
assert np.isfinite(test_metrics["accuracy"])
assert len(submission_numeric) == len(submission_input)

print("QA final automático superado.")
print("Notebook listo para Save Version después de completar las revisiones manuales.")
print("Archivo de envío:", NUMERIC_PATH)